In [2]:
import duckdb
import pandas as pd

In [11]:
show_table_sql = '''
show tables
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(show_table_sql).df())

,name
0,bronze_parking_violation_codes
1,bronze_parking_violations
2,bronze_parking_violations_codes
3,first_model
4,gold_ticket_metrics
5,gold_vehicles_metrics
6,parking_violation_2023
7,parking_violation_codes
8,ref_model
9,silver_parking_violation_codes


In [ ]:
sql_query_1 = '''
create or replace table parking_violation_codes as
select * from read_csv_auto(
    'data/dof_parking_violation_codes.csv',
    normalize_names=True
)
'''

sql_query_2 = '''
create or replace table parking_violation_2023 as
select * from read_csv_auto(
    'data/parking_violations_issued_fiscal_year_2023_sample.csv',
    normalize_names=True
)
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
    con.sql(sql_query_1)
    con.sql(sql_query_2)


In [8]:
select_table_sql = '''
select * from parking_violation_codes limit 5
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(select_table_sql).df())

,code,definition,manhattan_96th_st_below,all_other_areas
0,1,FAILURE TO DISPLAY BUS PERMIT,515,515
1,2,NO OPERATOR NAM/ADD/PH DISPLAY,515,515
2,3,UNAUTHORIZED PASSENGER PICK-UP,515,515
3,4,BUS PARKING IN LOWER MANHATTAN,115,115
4,5,BUS LANE VIOLATION,250,250


In [10]:
sql_ref_model = '''
select * from ref_model
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(sql_ref_model).df())

,count_star()
0,97


In [ ]:
sql_first_model = '''
select count(*) from first_model
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(sql_first_model).df())

,count_star()
0,97


In [17]:
sql_silver = '''
SELECT
    violation_code,
    SUM(fee_usd) AS total_revenue_usd
FROM silver_parking_violation_codes
GROUP BY
    violation_code
--HAVING
--    NOT(total_revenue_usd >= 1)
order by total_revenue_usd
limit 10
'''

# 7	silver_parking_violation_codes
# 8	silver_parking_violations
# 9	silver_violation_tickets
# 10	silver_violation_vehicles


with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(sql_silver).df())


,violation_code,total_revenue_usd
0,41,0.0
1,90,65.0
2,7,100.0
3,32,100.0
4,34,100.0
5,35,100.0
6,36,100.0
7,37,100.0
8,38,100.0
9,42,100.0


In [ ]:
# 4	gold_ticket_metrics
# 5	gold_vehicles_metrics

sql_gold = '''
select * from gold_vehicles_metrics
limit 10
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(sql_gold).df())

,registration_state,ticket_count
0,NJ,9258
1,PA,3514
2,FL,2414
3,CT,1787
4,GA,840
5,VA,797
6,MA,788
7,IN,765
8,NC,614
9,TX,594


In [18]:
sql_test_failure_query = '''
select * from "nyc_parking_violations"."main_dbt_test__audit"."violation_codes_revenue"
'''

with duckdb.connect('data/nyc_parking_violations.db') as con:
  display(con.sql(sql_test_failure_query).df())


,violation_code,total_revenue_usd
0,41,0.0
